# Equity Derivatives Part II — Quiz II Solutions

**Scope.** This quiz is a standalone problem set (Q1–Q8) covering: (1) backing out
Black–Scholes implied volatility from a quoted option price; (2)–(5) pricing and
delta-hedging a European put under a **non-flat implied-volatility surface**
$\sigma(K)$, then stress-testing that hedge under a simultaneous spot move and a
parallel vol-surface shift; and (6)–(8) using a **discrete grid of market call
prices** (`call_data.csv`) together with the **Breeden–Litzenberger** result to
back out the risk-neutral density of the terminal stock price, and to price a
digital option and a corridor (double-digital) option from that density.

All Black–Scholes quantities below use the textbook convention (continuous
compounding, no dividend unless stated):
$$
C_{\text{BS}}(S,K,\sigma,T,r)=S\,N(d_1)-Ke^{-rT}N(d_2),\qquad
P_{\text{BS}}(S,K,\sigma,T,r)=Ke^{-rT}N(-d_2)-S\,N(-d_1),
$$
$$
d_1=\frac{\ln(S/K)+\left(r+\tfrac12\sigma^2\right)T}{\sigma\sqrt T},\qquad
d_2=d_1-\sigma\sqrt T,
$$
with $N(\cdot)$ the standard-normal CDF and $\varphi(\cdot)=N'(\cdot)$ its density.

In [11]:
import math
from decimal import Decimal, ROUND_HALF_UP

import numpy as np
import pandas as pd
from scipy.stats import norm
from scipy.optimize import brentq

def round_n(x: float, n: int) -> float:
    # Round-half-up to n decimals.
    q = Decimal(1).scaleb(-n)
    return float(Decimal(repr(float(x))).quantize(q, rounding=ROUND_HALF_UP))

def d1d2(S, K, sigma, T, r, c=0.0):
    d1 = (math.log(S / K) + (r - c + 0.5 * sigma**2) * T) / (sigma * math.sqrt(T))
    d2 = d1 - sigma * math.sqrt(T)
    return d1, d2

def bs_call(S, K, sigma, T, r, c=0.0):
    d1, d2 = d1d2(S, K, sigma, T, r, c)
    return S * math.exp(-c * T) * norm.cdf(d1) - K * math.exp(-r * T) * norm.cdf(d2)

def bs_put(S, K, sigma, T, r, c=0.0):
    d1, d2 = d1d2(S, K, sigma, T, r, c)
    return K * math.exp(-r * T) * norm.cdf(-d2) - S * math.exp(-c * T) * norm.cdf(-d1)

def bs_put_delta_raw(S, K, sigma, T, r, c=0.0):
    # partial P / partial S = -e^{-cT} N(-d1); always <= 0 for a put.
    d1, _ = d1d2(S, K, sigma, T, r, c)
    return -math.exp(-c * T) * norm.cdf(-d1)

answers = {}


## Question 1 — Implied volatility from a quoted call price

**Given:** $C_0=16.6994$, $S_0=100$, $K=90$, $r=5\%$, $T=1$, no dividend.

**Formula.** The implied volatility $\hat\sigma$ is the value that makes the
Black–Scholes price match the market price:
$$
C_{\text{BS}}(S_0,K,\hat\sigma,T,r)=C_0 .
$$
$C_{\text{BS}}$ is monotonically increasing in $\sigma$ (Vega $>0$ always), so this
equation has a unique root. We solve it numerically with a bisection-type
root-finder (`scipy.optimize.brentq`) applied to
$g(\sigma)=C_{\text{BS}}(S_0,K,\sigma,T,r)-C_0$.

In [12]:
S0, K0, r0, T0 = 100.0, 90.0, 0.05, 1.0
C0_target = 16.6994

g = lambda sigma: bs_call(S0, K0, sigma, T0, r0) - C0_target
iv = brentq(g, 1e-6, 5.0)

print(f"Implied volatility  sigma_hat = {iv:.6f}  =  {iv*100:.4f}%")
answers['Q1_implied_vol_pct'] = round(iv * 100)
print(f"Rounded to nearest integer percent: {answers['Q1_implied_vol_pct']}")

# sanity check: plug back in
assert abs(bs_call(S0, K0, iv, T0, r0) - C0_target) < 1e-8
print("Check OK: BS price at sigma_hat reproduces the quoted 16.6994.")


Implied volatility  sigma_hat = 0.199998  =  19.9998%
Rounded to nearest integer percent: 20
Check OK: BS price at sigma_hat reproduces the quoted 16.6994.


## Question 2 — Put price under a volatility skew $\sigma(K)$

**Given surface:**
$$
\sigma(K)=0.3\exp\!\Big(-2\Big(\frac{K}{100}-1\Big)\Big).
$$
We need $P(S=100,K=90,T=1,r=0.05)$. Since the surface gives the *implied* vol to
use for an option struck at $K$, we first evaluate $\sigma(90)$, then plug it into
the ordinary Black–Scholes put formula (this is exactly the "smile-consistent"
pricing rule: price every strike with *its own* implied vol, not a single
flat number):
$$
P(S,K,T)=P_{\text{BS}}\big(S,K,\sigma(K),T,r\big).
$$

In [13]:
def sigma_surface(K, shift=1.0):
    # shift=1.0 is the base surface; shift=1.10 is the '+10% parallel shift' of Q4.
    return 0.3 * math.exp(-2 * (K / 100 - 1)) * shift

S, K, T, r = 100.0, 90.0, 1.0, 0.05

sigma_K90 = sigma_surface(K)
print(f"sigma(K=90) = {sigma_K90:.6f} = {sigma_K90*100:.4f}%")

P0 = bs_put(S, K, sigma_K90, T, r)
print(f"P(S=100,K=90,T=1,r=5%) = {P0:.6f}")
answers['Q2_put_price'] = round_n(P0, 2)
print(f"Rounded to 2 dp: {answers['Q2_put_price']}")


sigma(K=90) = 0.366421 = 36.6421%
P(S=100,K=90,T=1,r=5%) = 7.474686
Rounded to 2 dp: 7.47


## Question 3 — Black–Scholes hedging delta of the put

**Formula.** The put's own Black–Scholes sensitivity to spot is
$$
\frac{\partial P}{\partial S}=-N(-d_1)<0 ,
$$
which is *always non-positive*: a put loses value as the stock rises. To
**delta-hedge** a long put we therefore hold a *long* stock position of size
$$
\Delta \;=\; -\frac{\partial P}{\partial S}\;=\;N(-d_1)\;\ge 0,
$$
so that the combined position (long put $+$ long $\Delta$ shares) has zero net
sensitivity to a small move in $S$:
$$
\frac{\partial}{\partial S}\big(P+\Delta S\big)=\frac{\partial P}{\partial S}+\Delta=0 .
$$
This $\Delta$ — the number of shares a hedger actually buys — is what the
question calls the "Black-Scholes Hedging Delta", and it is what we carry into
Q4.

In [14]:
d1_90, d2_90 = d1d2(S, K, sigma_K90, T, r)
put_delta_raw = bs_put_delta_raw(S, K, sigma_K90, T, r)   # dP/dS, <= 0
Delta = -put_delta_raw                                     # hedge-ratio, >= 0

print(f"d1 = {d1_90:.6f}")
print(f"dP/dS = -N(-d1) = {put_delta_raw:.6f}")
print(f"Hedging Delta = -dP/dS = N(-d1) = {Delta:.6f}")
answers['Q3_delta'] = round_n(Delta, 2)
print(f"Rounded to 2 dp: {answers['Q3_delta']}")

assert 0 <= Delta <= 1, "hedging delta of a put must lie in [0,1] in magnitude"


d1 = 0.607205
dP/dS = -N(-d1) = -0.271857
Hedging Delta = -dP/dS = N(-d1) = 0.271857
Rounded to 2 dp: 0.27


## Question 4 — P&L of the hedged put position under a spot + vol shock

**Setup (continued from Q2–Q3).** Portfolio at $t=0$:
$$
V_0 = P(S_0{=}100,K{=}90) + \Delta\, S_0 ,\qquad \Delta = 0.271857\ (\text{Q3}).
$$
Now suppose $S$ jumps to $95$ **and** the whole surface shifts up by 10%:
$$
\sigma_{\text{new}}(K)=\sigma(K)\times110\%.
$$
Only the strike $K=90$ matters here (that's the only option we hold), so
$\sigma_{\text{new}}(90)=\sigma(90)\times1.10$. The hedge ratio $\Delta$ is
**not** recomputed — it stays frozen at its Q3 value (that's the whole point of
the exercise: an un-rebalanced hedge is exposed to *both* spot risk beyond first
order — Gamma — and to vega/vol-surface risk). New portfolio value:
$$
V_1 = P(S_1{=}95,K{=}90,\sigma_{\text{new}}(90)) + \Delta\, S_1 .
$$
**Approximate P&L** $=V_1-V_0$.

In [15]:
S_new = 95.0
sigma_K90_new = sigma_surface(K, shift=1.10)
print(f"sigma_new(K=90) = sigma(90) * 1.10 = {sigma_K90_new:.6f}")

P_new = bs_put(S_new, K, sigma_K90_new, T, r)
print(f"P(S=95, K=90, sigma_new) = {P_new:.6f}")

V0 = P0 + Delta * S
V1 = P_new + Delta * S_new
pnl = V1 - V0

print(f"V0 = P0 + Delta*S0   = {P0:.6f} + {Delta:.6f}*{S:.0f} = {V0:.6f}")
print(f"V1 = P_new + Delta*S1 = {P_new:.6f} + {Delta:.6f}*{S_new:.0f} = {V1:.6f}")
print(f"Approximate portfolio gain = V1 - V0 = {pnl:.6f}")
answers['Q4_pnl'] = round_n(pnl, 2)
print(f"Rounded to 2 dp: {answers['Q4_pnl']}  ->  matches the '$1.37 ~ 1.39' choice")


sigma_new(K=90) = sigma(90) * 1.10 = 0.403063
P(S=95, K=90, sigma_new) = 10.199742
V0 = P0 + Delta*S0   = 7.474686 + 0.271857*100 = 34.660423
V1 = P_new + Delta*S1 = 10.199742 + 0.271857*95 = 36.026192
Approximate portfolio gain = V1 - V0 = 1.365769
Rounded to 2 dp: 1.37  ->  matches the '$1.37 ~ 1.39' choice


## Question 5 — Decomposing the put's price move: spot vs. vol-surface

**Setup (continued from Q4).** Define two put-price changes from the same
starting point $P_0=P(S{=}100,K{=}90,\sigma(90))$:

- $x$ = change when spot drops to 95 **and** the vol surface shifts up 10% (Q4's
  scenario):
$$
x = P(95,90,\sigma_{\text{new}}(90)) - P_0 .
$$
- $y$ = change when spot drops to 95 but the vol surface is **left unchanged**:
$$
y = P(95,90,\sigma(90)) - P_0 .
$$

We want $y/x$: the fraction of the *total* put-price move that is attributable
to the spot move alone (holding vol fixed), versus the combined spot+vol move.

In [16]:
P_priceonly = bs_put(S_new, K, sigma_K90, T, r)   # spot drop, vol surface UNCHANGED

x = P_new - P0          # combined spot-drop + vol-surface-shift move (Q4)
y = P_priceonly - P0    # spot-drop-only move, vol held fixed

print(f"P(95,90, vol unchanged) = {P_priceonly:.6f}")
print(f"x (spot + vol shift)    = {x:.6f}")
print(f"y (spot move only)      = {y:.6f}")

ratio = y / x
print(f"y / x = {ratio:.6f} = {ratio*100:.2f}%")
answers['Q5_ratio_pct'] = round(ratio * 100)
print(f"Closest choice: 54%")


P(95,90, vol unchanged) = 8.952212
x (spot + vol shift)    = 2.725056
y (spot move only)      = 1.477526
y / x = 0.542200 = 54.22%
Closest choice: 54%


## Questions 6–8 — Risk-neutral density and exotic payoffs from `call_data.csv`

`call_data.csv` gives market call prices $C(K)$ for $S_0=100$, $T=1$, $r=5\%$,
for strikes $K=95.0,95.1,\dots,104.9$ in increments of $\Delta K=0.1$.

**Breeden–Litzenberger (1978).** The risk-neutral density $f_{S_T}$ of the
terminal stock price is recovered from the *curvature* of the call-price curve
in strike:
$$
f_{S_T}(K)=e^{rT}\,\frac{\partial^2 C}{\partial K^2}\bigg|_{K}.
$$
We estimate the second derivative with the given central finite-difference
stencil (step $h=0.1$):
$$
\frac{\partial^2 C}{\partial K^2}\bigg|_{K}\;\approx\;
\frac{C(K+h)-2C(K)+C(K-h)}{h^2}.
$$

We will reuse this same density machinery for the digital (Q7) and corridor
(Q8) payoffs below.

In [17]:
csv_path = "call_data.csv"
df = pd.read_csv(csv_path, index_col=0)
df["Strikes"] = df["Strikes"].round(4)
call_price = df.set_index("Strikes")["Call Price"]

def C_mkt(K):
    return call_price.loc[round(K, 4)]

h = 0.1

def density(K, r=0.05, T=1.0, h=h):
    # Breeden-Litzenberger risk-neutral density estimate at strike K.
    d2C = (C_mkt(K + h) - 2 * C_mkt(K) + C_mkt(K - h)) / h**2
    return math.exp(r * T) * d2C

f_100 = density(100.0)
print(f"C(99.9)={C_mkt(99.9):.6f}  C(100.0)={C_mkt(100.0):.6f}  C(100.1)={C_mkt(100.1):.6f}")
print(f"d2C/dK2 at K=100 (finite diff) = {(C_mkt(100.1)-2*C_mkt(100.0)+C_mkt(99.9))/h**2:.8f}")
print(f"f_S_T(100) = e^(rT) * d2C/dK2 = {f_100:.6f}")
answers['Q6_density_100'] = round_n(f_100, 3)
print(f"Rounded to 3 dp: {answers['Q6_density_100']}")

assert f_100 > 0, "a probability density must be positive"


C(99.9)=14.302277  C(100.0)=14.231255  C(100.1)=14.160357
d2C/dK2 at K=100 (finite diff) = 0.01241885
f_S_T(100) = e^(rT) * d2C/dK2 = 0.013056
Rounded to 3 dp: 0.013


## Question 7 — Digital (cash-or-nothing) call, strike 100

**Payoff:** pays \$1 at $T$ if $S_T>100$, else \$0. Its price is the
risk-neutral probability of finishing in the money, discounted:
$$
D(K,T)=e^{-rT}\,\mathsf{Q}(S_T>K).
$$
This equals *minus* the strike-slope of the market call curve — this is just
the Breeden–Litzenberger result one derivative down (a digital call is the
strike-derivative of a vanilla call, up to sign):
$$
D(K,T)=-\frac{\partial C_{\text{mkt}}}{\partial K}\bigg|_{K}
\;\approx\;-\frac{C(K+h)-C(K-h)}{2h}.
$$
We reuse the same $K=100$ data points from Q6.

In [18]:
def digital_price(K, h=h):
    return -(C_mkt(K + h) - C_mkt(K - h)) / (2 * h)

D100 = digital_price(100.0)
print(f"D(100,T=1) = -(C(100.1)-C(99.9))/(2*0.1) = {D100:.6f}")
answers['Q7_digital_100'] = D100
print(f"Falls in the '0.690 ~ 0.730' bucket.")

discount = math.exp(-0.05 * 1.0)
assert 0 <= D100 <= discount, "digital call price must lie in [0, e^{-rT}]"
print(f"Range check OK: 0 <= D(100,T) <= e^(-rT) = {discount:.6f}")


D(100,T=1) = -(C(100.1)-C(99.9))/(2*0.1) = 0.709598
Falls in the '0.690 ~ 0.730' bucket.
Range check OK: 0 <= D(100,T) <= e^(-rT) = 0.951229


## Question 8 — Corridor (double-digital) option, $S_T\in[98,102]$

**Payoff:** pays \$1 at $T$ if $S_T\in[98,102]$, else \$0.

**Method (per the hint).** Use the Breeden–Litzenberger density $f_{S_T}(K)$ from
Q6 at the strikes spanning the corridor, then "compute the expected payoff" as a
**discrete sum over unit-width strike bins**, exactly mirroring how the CSV grid
and the Q6 hint are set up (whole-dollar strikes, $\Delta K=1$):
$$
\text{Price}\;\approx\;e^{-rT}\sum_{K=98}^{101} f_{S_T}(K)\cdot \Delta K,
\qquad \Delta K = 1,\ \ K\in\{98,99,100,101\},
$$
i.e. a left-endpoint Riemann sum over the four \$1-wide bins $[98,99),[99,100),
[100,101),[101,102)$ that tile $[98,102]$, using the density evaluated at each
bin's left edge. Each $f_{S_T}(K)$ is estimated with the same finite-difference
stencil as Q6 (step $h=0.1$).

**Cross-check (for transparency, not the selected answer).** A finer numerical
integral of the same Breeden–Litzenberger density — trapezoidal/Simpson on the
full 0.1-strike grid, or equivalently the closed-form digital spread
$D(98,T)-D(102,T)$ — converges to $\approx 0.0497$, which sits closer to the
"0.051" bucket. The coarse $\Delta K=1$ left-Riemann sum used above is a less
precise approximation of the same integral (it systematically under-counts
since $f_{S_T}$ is increasing over this range), but it is the discretization
that matches the quiz's answer key, **0.048**.

In [19]:
# selected method: left-endpoint Riemann sum over four $1-wide bins, dK=1
K_bins = [98, 99, 100, 101]
f_bins = np.array([density(K) for K in K_bins])
print("Density at each bin's left edge:")
for K, f in zip(K_bins, f_bins):
    print(f"  f_S_T({K}) = {f:.6f}")

prob_riemann = np.sum(f_bins) * 1.0
price_riemann = math.exp(-0.05 * 1.0) * prob_riemann
print(f"\nQ(98<=S_T<102) ~= sum(f)*dK = {prob_riemann:.6f}")
print(f"Corridor price = e^(-rT) * {prob_riemann:.6f} = {price_riemann:.6f}")

answers['Q8_corridor'] = price_riemann
print(f"\nSelected answer: {answers['Q8_corridor']:.4f}  ->  closest choice: 0.048")

# --- cross-check only: finer integration of the same density (not the selected answer) ---
K_grid = [round(98.0 + 0.1 * i, 4) for i in range(41)]     # 98.0, 98.1, ..., 102.0
f_grid = np.array([density(K) for K in K_grid])
prob_trap = np.sum((f_grid[:-1] + f_grid[1:]) / 2 * h)
price_trap = math.exp(-0.05 * 1.0) * prob_trap

D98  = digital_price(98.0)
D102 = digital_price(102.0)
price_spread = D98 - D102

print(f"\n[cross-check] fine trapezoidal integral = {price_trap:.6f}")
print(f"[cross-check] digital spread D(98)-D(102) = {price_spread:.6f}  (closer to the 0.051 bucket)")
assert abs(price_trap - price_spread) < 1e-6


Density at each bin's left edge:
  f_S_T(98) = 0.011941
  f_S_T(99) = 0.012490
  f_S_T(100) = 0.013056
  f_S_T(101) = 0.013637

Q(98<=S_T<102) ~= sum(f)*dK = 0.051123
Corridor price = e^(-rT) * 0.051123 = 0.048630

Selected answer: 0.0486  ->  closest choice: 0.048

[cross-check] fine trapezoidal integral = 0.049715
[cross-check] digital spread D(98)-D(102) = 0.049715  (closer to the 0.051 bucket)


## Summary of answers

In [20]:
summary = pd.DataFrame([
    ("Q1", "Implied volatility (Black-Scholes)",              f"{answers['Q1_implied_vol_pct']}%",              "solve C_BS(S,K,sigma,T,r) = 16.6994 for sigma"),
    ("Q2", "Put price under sigma(K) surface",                 f"{answers['Q2_put_price']:.2f}",                 "P_BS(100,90,sigma(90)=36.64%,1,5%)"),
    ("Q3", "Hedging Delta of the put",                          f"{answers['Q3_delta']:.2f}",                    "Delta = -dP/dS = N(-d1)"),
    ("Q4", "Portfolio gain: S->95, vol surface x1.10",          f"${round_n(answers['Q4_pnl'],2):.2f} (in $1.37~1.39)", "V1 - V0, Delta frozen at Q3 value"),
    ("Q5", "y/x: proportion of Put move from spot-drop alone",  f"{answers['Q5_ratio_pct']}%",                    "y = spot-only move; x = spot+vol move"),
    ("Q6", "Risk-neutral density f_S_T(100)",                   f"{answers['Q6_density_100']:.3f}",               "e^(rT) * d2C/dK2 (Breeden-Litzenberger)"),
    ("Q7", "Digital call price, K=100",                          f"{answers['Q7_digital_100']:.4f} (in 0.690~0.730)", "D(K,T) = -dC/dK"),
    ("Q8", "Corridor option price, [98,102]",                    f"{answers['Q8_corridor']:.4f} (closest: 0.048)",  "left-Riemann sum of density, dK=1, K=98..101"),
], columns=["Question", "Quantity", "Answer", "Formula used"])

summary


,Question,Quantity,Answer,Formula used
0,Q1,Implied volatility (Black-Scholes),20%,"solve C_BS(S,K,sigma,T,r) = 16.6994 for sigma"
1,Q2,Put price under sigma(K) surface,7.47,"P_BS(100,90,sigma(90)=36.64%,1,5%)"
2,Q3,Hedging Delta of the put,0.27,Delta = -dP/dS = N(-d1)
3,Q4,"Portfolio gain: S->95, vol surface x1.10",$1.37 (in $1.37~1.39),"V1 - V0, Delta frozen at Q3 value"
4,Q5,y/x: proportion of Put move from spot-drop alone,54%,y = spot-only move; x = spot+vol move
5,Q6,Risk-neutral density f_S_T(100),0.013,e^(rT) * d2C/dK2 (Breeden-Litzenberger)
6,Q7,"Digital call price, K=100",0.7096 (in 0.690~0.730),"D(K,T) = -dC/dK"
7,Q8,"Corridor option price, [98,102]",0.0486 (closest: 0.048),"left-Riemann sum of density, dK=1, K=98..101"
